# Trees and Graphs

Notebook for **Chapter 10** of the *cs* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/cs/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 Binary Trees and the Four Traversals

In [ ]:
from collections import deque

class Node:
    def __init__(s, v, l=None, r=None):
        s.val = v; s.left = l; s.right = r

def insert(root, v):
    if root is None: return Node(v)
    if v < root.val: root.left = insert(root.left, v)
    else:            root.right = insert(root.right, v)
    return root

root = None
for v in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    root = insert(root, v)

def inorder(n):   return inorder(n.left) + [n.val] + inorder(n.right) if n else []
def preorder(n):  return [n.val] + preorder(n.left) + preorder(n.right) if n else []
def postorder(n): return postorder(n.left) + postorder(n.right) + [n.val] if n else []

def level(n):
    out = []; q = deque([n])
    while q:
        row = []
        for _ in range(len(q)):          # exactly one level per pass
            x = q.popleft(); row.append(x.val)
            if x.left:  q.append(x.left)
            if x.right: q.append(x.right)
        out.append(row)
    return out

def height(n): return 0 if n is None else 1 + max(height(n.left), height(n.right))

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
def height(n): return 1 + max(height(n.left), height(n.right))   # no "if n is None"
height(Node(1))

AttributeError: 'NoneType' object has no attribute 'left'


## 10.2 Binary Search Trees and Lowest Common Ancestor

In [ ]:
def search(n, t):
    steps = 0
    while n:
        steps += 1
        if t == n.val: return True, steps
        n = n.left if t < n.val else n.right
    return False, steps

def lca(n, a, b):
    while n:
        if a < n.val and b < n.val:   n = n.left
        elif a > n.val and b > n.val: n = n.right
        else: return n.val

def is_bst(n, lo=float('-inf'), hi=float('inf')):
    if not n: return True
    return lo < n.val < hi and is_bst(n.left, lo, n.val) and is_bst(n.right, n.val, hi)

## 10.3 Tries (Prefix Trees)

In [ ]:
class Trie:
    def __init__(s): s.kids = {}; s.end = False
    def add(s, w):
        n = s
        for c in w: n = n.kids.setdefault(c, Trie())
        n.end = True
    def starts(s, p):
        n = s
        for c in p:
            if c not in n.kids: return []
            n = n.kids[c]
        out = []
        def go(n, path):
            if n.end: out.append(p + path)
            for c in sorted(n.kids): go(n.kids[c], path + c)
        go(n, ""); return out

t = Trie()
for w in ["car", "card", "care", "cat", "dog", "do"]: t.add(w)
print(t.starts("car"), t.starts("do"), t.starts("x"))

## 10.4 Graphs: BFS, DFS and Connected Components

In [ ]:
from collections import deque

g = {'A':['B','C'], 'B':['A','D'], 'C':['A','D','E'],
     'D':['B','C','F'], 'E':['C','F'], 'F':['D','E']}

def bfs(g, s):
    dist = {s: 0}; q = deque([s]); order = []
    while q:
        u = q.popleft(); order.append(u)
        for v in g[u]:
            if v not in dist:           # "seen" check prevents infinite loops
                dist[v] = dist[u] + 1; q.append(v)
    return order, dist

def dfs(g, u, seen=None, order=None):
    seen = seen if seen is not None else set()
    order = order if order is not None else []
    seen.add(u); order.append(u)
    for v in g[u]:
        if v not in seen: dfs(g, v, seen, order)
    return order

g2 = {1:[2], 2:[1,3], 3:[2], 4:[5], 5:[4], 6:[]}
def comps(g):
    seen = set(); res = []
    for s in g:
        if s not in seen:
            res.append(sorted(dfs(g, s, seen, [])))
    return res

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
def dfs_bad(g, u, order):
    order.append(u)
    for v in g[u]: dfs_bad(g, v, order)     # A -> B -> A -> B ...

RecursionError: maximum recursion depth exceeded


## 10.5 Topological Sort

In [ ]:
deps = {'extract':['clean'], 'clean':['features','report'],
        'features':['train'], 'train':['report'], 'report':[]}

def topo(g):
    indeg = {u: 0 for u in g}
    for u in g:
        for v in g[u]: indeg[v] += 1
    q = deque(sorted(u for u in g if indeg[u] == 0)); out = []
    while q:
        u = q.popleft(); out.append(u)
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0: q.append(v)
    return out if len(out) == len(g) else None

print(topo(deps))
deps['report'].append('extract')      # report -> extract creates a cycle
print("cycle:", topo(deps))

## 10.6 Dijkstra's Shortest Path

In [ ]:
import heapq
w = {'A':{'B':4,'C':1}, 'B':{'D':1}, 'C':{'B':2,'D':5}, 'D':{}}

def dijk(w, s):
    d = {u: float('inf') for u in w}; d[s] = 0
    pq = [(0, s)]; prev = {}
    while pq:
        du, u = heapq.heappop(pq)
        if du > d[u]: continue            # stale entry
        for v, c in w[u].items():
            if du + c < d[v]:
                d[v] = du + c; prev[v] = u
                heapq.heappush(pq, (d[v], v))
    return d, prev

d, prev = dijk(w, 'A'); print(d)
p = ['D']
while p[-1] != 'A': p.append(prev[p[-1]])
print(p[::-1])

## 10.7 Union-Find (Disjoint Set)

In [ ]:
class UF:
    def __init__(s, n): s.p = list(range(n)); s.r = [0]*n
    def find(s, x):
        while s.p[x] != x:
            s.p[x] = s.p[s.p[x]]      # path compression (halving)
            x = s.p[x]
        return x
    def union(s, a, b):
        a, b = s.find(a), s.find(b)
        if a == b: return False       # already same group
        if s.r[a] < s.r[b]: a, b = b, a
        s.p[b] = a
        if s.r[a] == s.r[b]: s.r[a] += 1
        return True

u = UF(6)
print([u.union(a, b) for a, b in [(0,1),(1,2),(0,2),(3,4)]])
print(u.find(0) == u.find(2), u.find(0) == u.find(3), len({u.find(i) for i in range(6)}))

## 10.8 Minimum Spanning Tree (Kruskal and Prim)

In [ ]:
from collections import defaultdict
edges = [(7,'A','B'),(5,'A','D'),(8,'B','C'),(9,'B','D'),(7,'B','E'),(5,'C','E'),
         (15,'D','E'),(6,'D','F'),(8,'E','F'),(9,'E','G'),(11,'F','G')]
idx = {c: i for i, c in enumerate("ABCDEFG")}

uf = UF(7); tot = 0; used = []             # Kruskal
for c, a, b in sorted(edges):
    if uf.union(idx[a], idx[b]): tot += c; used.append((a, b, c))
print(used, tot)

adj = defaultdict(list)                     # Prim
for c, a, b in edges: adj[a].append((c, b)); adj[b].append((c, a))
seen = {'A'}; pq = list(adj['A']); heapq.heapify(pq); tot = 0; used = []
while pq and len(seen) < 7:
    c, v = heapq.heappop(pq)
    if v in seen: continue
    seen.add(v); tot += c; used.append((v, c))
    for e in adj[v]:
        if e[1] not in seen: heapq.heappush(pq, e)
print(used, tot)

## 10.9 Range Queries: Fenwick Tree, Segment Tree and Sparse Table

In [ ]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)          # 1-indexed; t[0] is unused

    def update(self, i, delta):         # add delta at position i (1-indexed)
        while i <= self.n:
            self.t[i] += delta
            i += i & -i                 # climb to the next entry that covers i

    def prefix(self, i):                # sum of positions 1..i
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i                 # drop the lowest set bit
        return s

    def range_sum(self, l, r):          # positions l..r inclusive
        return self.prefix(r) - self.prefix(l - 1)

a = [5, 3, 7, 9, 6, 4, 1, 2]
f = Fenwick(8)
for pos, v in enumerate(a, start=1):
    f.update(pos, v)
print("t           ", f.t[1:])
print("prefix(7)   ", f.prefix(7))
print("range 3..6  ", f.range_sum(3, 6))
f.update(3, 4)                           # a[3] goes from 7 to 11
print("t after +4  ", f.t[1:])
print("range 3..6  ", f.range_sum(3, 6))

t            [5, 8, 7, 24, 6, 10, 1, 37]
prefix(7)    35
range 3..6   26
t after +4   [5, 8, 11, 28, 6, 10, 1, 41]
range 3..6   30


In [ ]:
class Fenwick:
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
    def update(self, i, d):
        while i <= self.n:
            self.t[i] += d; i += i & -i
    def prefix(self, i):
        s = 0
        while i > 0:
            s += self.t[i]; i -= i & -i
        return s

def inversions(a):
    f = Fenwick(max(a))
    total = 0
    for seen, x in enumerate(a):         # seen = how many values came before x
        total += seen - f.prefix(x)      # earlier values that are larger than x
        f.update(x, 1)
    return total

print(inversions([3, 1, 2, 5, 4]))
print(inversions([5, 4, 3, 2, 1]))      # fully reversed: 5*4/2 = 10
print(inversions([1, 2, 3, 4]))         # sorted: 0

3
10
0


In [ ]:
class SegTree:
    def __init__(self, a, op=lambda x, y: x + y, neutral=0):
        self.n = len(a); self.op = op; self.neutral = neutral
        self.t = [neutral] * self.n + list(a)          # leaves live at n..2n-1
        for i in range(self.n - 1, 0, -1):
            self.t[i] = op(self.t[2 * i], self.t[2 * i + 1])

    def update(self, i, v):                            # set a[i] = v
        i += self.n; self.t[i] = v
        while i > 1:
            i //= 2
            self.t[i] = self.op(self.t[2 * i], self.t[2 * i + 1])

    def query(self, l, r, show=False):                 # combine a[l:r], half-open
        res = self.neutral; l += self.n; r += self.n
        while l < r:
            if l & 1:
                if show: print("  take node", l, "=", self.t[l])
                res = self.op(res, self.t[l]); l += 1
            if r & 1:
                r -= 1
                if show: print("  take node", r, "=", self.t[r])
                res = self.op(res, self.t[r])
            l //= 2; r //= 2
        return res

a = [5, 3, 7, 9, 6, 4, 1, 2]
s = SegTree(a)
print("tree array", s.t[1:])
print("sum a[2:6]", s.query(2, 6, show=True))
s.update(2, 11)
print("after a[2]=11, sum a[2:6]", s.query(2, 6))
m = SegTree(a, op=min, neutral=float("inf"))
print("min a[1:6]", m.query(1, 6))

tree array [37, 24, 13, 8, 16, 10, 3, 5, 3, 7, 9, 6, 4, 1, 2]
  take node 5 = 16
  take node 6 = 10
sum a[2:6] 26
after a[2]=11, sum a[2:6] 30
min a[1:6] 3


In [ ]:
import random

class LazySeg:
    def __init__(self, a):
        self.n = len(a); self.s = [0] * (4 * self.n); self.z = [0] * (4 * self.n)
        self._build(1, 0, self.n - 1, a)
    def _build(self, k, lo, hi, a):
        if lo == hi: self.s[k] = a[lo]; return
        mid = (lo + hi) // 2
        self._build(2 * k, lo, mid, a); self._build(2 * k + 1, mid + 1, hi, a)
        self.s[k] = self.s[2 * k] + self.s[2 * k + 1]
    def _push(self, k, lo, hi):
        if self.z[k]:
            mid = (lo + hi) // 2
            for c, clo, chi in ((2 * k, lo, mid), (2 * k + 1, mid + 1, hi)):
                self.s[c] += self.z[k] * (chi - clo + 1); self.z[c] += self.z[k]
            self.z[k] = 0
    def add(self, l, r, v, k=1, lo=0, hi=None):        # inclusive l..r
        hi = self.n - 1 if hi is None else hi
        if r < lo or hi < l: return
        if l <= lo and hi <= r:
            self.s[k] += v * (hi - lo + 1); self.z[k] += v; return
        self._push(k, lo, hi); mid = (lo + hi) // 2
        self.add(l, r, v, 2 * k, lo, mid); self.add(l, r, v, 2 * k + 1, mid + 1, hi)
        self.s[k] = self.s[2 * k] + self.s[2 * k + 1]
    def sum(self, l, r, k=1, lo=0, hi=None):
        hi = self.n - 1 if hi is None else hi
        if r < lo or hi < l: return 0
        if l <= lo and hi <= r: return self.s[k]
        self._push(k, lo, hi); mid = (lo + hi) // 2
        return self.sum(l, r, 2 * k, lo, mid) + self.sum(l, r, 2 * k + 1, mid + 1, hi)

random.seed(7)
a = [random.randint(0, 9) for _ in range(20)]
ref = a[:]; seg = LazySeg(a); bad = 0
for _ in range(300):
    l = random.randint(0, 19); r = random.randint(l, 19)
    if random.random() < 0.5:
        v = random.randint(1, 5); seg.add(l, r, v)
        for i in range(l, r + 1): ref[i] += v
    elif seg.sum(l, r) != sum(ref[l:r + 1]): bad += 1
print("mismatches:", bad)
b = LazySeg([5, 3, 7, 9, 6, 4, 1, 2]); b.add(0, 3, 5)
print("sum 0..7 after +5 on 0..3:", b.sum(0, 7), "(37 + 4*5 = 57)")

mismatches: 0
sum 0..7 after +5 on 0..3: 57 (37 + 4*5 = 57)


In [ ]:
a = [5, 3, 7, 9, 6, 4, 1, 2]
n = len(a)
st = [a[:]]                                       # st[0][i] = a[i]
k = 1
while (1 << k) <= n:
    prev = st[-1]
    st.append([min(prev[i], prev[i + (1 << (k - 1))]) for i in range(n - (1 << k) + 1)])
    k += 1

def rmq(l, r):                                    # inclusive l..r
    k = (r - l + 1).bit_length() - 1              # floor(log2(length))
    return min(st[k][l], st[k][r - (1 << k) + 1])

print("levels:", len(st))
print("st[2]:", st[2])
print("min 1..5:", rmq(1, 5))
print("min 4..7:", rmq(4, 7))
print("min 0..7:", rmq(0, 7))

levels: 4
st[2]: [3, 3, 4, 1, 1]
min 1..5: 3
min 4..7: 1
min 0..7: 1


## 10.10 Strongly Connected Components and Bipartite Graphs

In [ ]:
import sys
sys.setrecursionlimit(10000)

def scc(n, edges):
    g = [[] for _ in range(n)]; rg = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); rg[v].append(u)
    seen = [False] * n; order = []
    def dfs1(u):
        seen[u] = True
        for v in g[u]:
            if not seen[v]: dfs1(v)
        order.append(u)                          # finish time order
    for u in range(n):
        if not seen[u]: dfs1(u)
    comp = [-1] * n; groups = []
    def dfs2(u, c):
        comp[u] = c; groups[c].append(u)
        for v in rg[u]:
            if comp[v] == -1: dfs2(v, c)
    for u in reversed(order):
        if comp[u] == -1:
            groups.append([]); dfs2(u, len(groups) - 1)
    return [sorted(x) for x in groups]

edges = [(0,1),(1,2),(2,0),(2,3),(3,4),(4,5),(5,3),(6,5),(6,7),(7,6)]
print(scc(8, edges))
print(scc(4, [(0,1),(1,2),(2,3)]))               # a chain: every node alone

[[6, 7], [0, 1, 2], [3, 4, 5]]
[[0], [1], [2], [3]]


In [ ]:
from collections import deque

def is_bipartite(n, edges):
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    color = [-1] * n
    for s in range(n):                           # one BFS per component
        if color[s] != -1: continue
        color[s] = 0; q = deque([s])
        while q:
            u = q.popleft()
            for v in g[u]:
                if color[v] == -1:
                    color[v] = 1 - color[u]; q.append(v)
                elif color[v] == color[u]:
                    return False
    return True

print("square  ", is_bipartite(4, [(0,1),(1,2),(2,3),(3,0)]))
print("triangle", is_bipartite(3, [(0,1),(1,2),(2,0)]))
print("both    ", is_bipartite(7, [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,4)]))

square   True
triangle False
both     False


## 10.11 Shortest Paths Beyond Dijkstra: Bellman-Ford, Floyd-Warshall, A*

Dijkstra (10.6) is fast but has two limits: edge weights cannot be negative, and it solves one source at a time. Three tools remove those limits. Every one of them is built on a single move called **relaxation**: if going through `u` to `v` is shorter than the best known way to `v`, update `dist[v] = dist[u] + w`.

In [ ]:
INF = float("inf")

def bellman_ford(n, edges, src):
    dist = [INF] * n; dist[src] = 0
    for rnd in range(1, n):                      # at most n-1 passes
        changed = False
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w; changed = True
        print(f"after pass {rnd}: {dist}")
        if not changed: break
    for u, v, w in edges:                         # one more look: still improving?
        if dist[u] + w < dist[v]:
            return None                           # negative cycle reachable
    return dist

A, B, C, D = range(4)
edges = [(A,B,4),(A,C,5),(B,C,-3),(C,D,3),(B,D,6)]
print(bellman_ford(4, edges, A))
print("--- add D->B with weight -2 (cycle B,C,D,B totals -3+3-2 = -2)")
print(bellman_ford(4, edges + [(D,B,-2)], A))

after pass 1: [0, 4, 1, 4]
after pass 2: [0, 4, 1, 4]
[0, 4, 1, 4]
--- add D->B with weight -2 (cycle B,C,D,B totals -3+3-2 = -2)
after pass 1: [0, 2, 1, 4]
after pass 2: [0, 0, -1, 2]
after pass 3: [0, -2, -3, 0]
None


In [ ]:
INF = float("inf")

def floyd(n, edges):
    d = [[0 if i == j else INF for j in range(n)] for i in range(n)]
    for u, v, w in edges: d[u][v] = min(d[u][v], w)
    for k in range(n):                            # k outermost
        for i in range(n):
            for j in range(n):
                if d[i][k] + d[k][j] < d[i][j]:
                    d[i][j] = d[i][k] + d[k][j]
    return d

edges = [(0,1,3),(0,3,10),(1,2,4),(2,3,2),(3,0,1)]
for row in floyd(4, edges): print(row)

[0, 3, 7, 9]
[7, 0, 4, 6]
[3, 6, 0, 2]
[1, 4, 8, 0]


In [ ]:
import heapq

N = 30
wall = {(r, 15) for r in range(0, 26)}
start, goal = (0, 0), (N - 1, N - 1)

def solve(use_h, walls, prefer_deep=False):
    h = (lambda p: abs(p[0] - goal[0]) + abs(p[1] - goal[1])) if use_h else (lambda p: 0)
    sign = -1 if prefer_deep else 1          # tie-break: prefer larger g when f is equal
    g = {start: 0}; pq = [(h(start), 0, start)]; expanded = 0; done = set()
    while pq:
        f, key, u = heapq.heappop(pq); d = g[u]
        if u in done: continue
        done.add(u); expanded += 1
        if u == goal: return d, expanded
        for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
            v = (u[0] + dr, u[1] + dc)
            if 0 <= v[0] < N and 0 <= v[1] < N and v not in walls and d + 1 < g.get(v, 10**9):
                g[v] = d + 1; heapq.heappush(pq, (d + 1 + h(v), sign * (d + 1), v))

print("wall  Dijkstra (h = 0):", solve(False, wall))
print("wall  A* Manhattan    :", solve(True, wall))
print("open  Dijkstra (h = 0):", solve(False, set()))
print("open  A* Manhattan    :", solve(True, set()))
print("open  A*, deep tie-break:", solve(True, set(), prefer_deep=True))

wall  Dijkstra (h = 0): (58, 643)
wall  A* Manhattan    : (58, 510)
open  Dijkstra (h = 0): (58, 900)
open  A* Manhattan    : (58, 900)
open  A*, deep tie-break: (58, 59)


## 10.12 Seeing the Algorithms: BFS Layers and Union-Find Forests

## Chapter recap

## Practice set

**1.** Pre-order of a tree with root 1, left child 2, right child 3? **Answer:** `[1, 2, 3]`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/cs/ch10.html